Import the two main datasets used in our analysis:
*    Pincode-Level Geographic Dataset
*    Amazon E-Commerce Sales Dataset



In [2]:
# --------------------------------------------------------------------------------
# Load required packages (install if missing)
# 'tidyverse' is a collection of packages for data manipulation (dplyr, ggplot2, etc.)
# --------------------------------------------------------------------------------
if (!require("tidyverse")) install.packages("tidyverse")
library(tidyverse)

# --------------------------------------------------------------------------------
# STEP 1: Read and clean the pincode-level location dataset
# This dataset contains geographic information for delivery areas across India.
# We select only the useful columns and remove rows with missing values.
# --------------------------------------------------------------------------------

CleanData <- read_csv("pin code data warehouse optimisation.csv", show_col_types = FALSE) %>%
  select(pincode, district, latitude, longitude) %>%  # Keep relevant columns
  drop_na()  # Remove rows with NA values

# --------------------------------------------------------------------------------
# STEP 2: Process the Amazon sales dataset to extract e-commerce demand
# The sales data is grouped by pincode, and total quantity sold is calculated
# to represent demand per pincode.
# --------------------------------------------------------------------------------

pincode_demand <- read.csv("Amazon Sale Report.csv") %>%
  group_by(ship.postal.code) %>%  # Group by shipping pincode
  summarise(Demand = sum(Qty, na.rm = TRUE)) %>%  # Total quantity = demand
  drop_na() %>%  # Remove incomplete records
  rename(pincode = ship.postal.code)  # Rename to match CleanData for joining

# --------------------------------------------------------------------------------
# STEP 3: Merge demand data with location data using pincode as the key
# This creates a unified dataset: pincode, district, lat, long, and demand.
# --------------------------------------------------------------------------------

final_data <- inner_join(CleanData, pincode_demand, by = "pincode")

# --------------------------------------------------------------------------------
# STEP 4: Save the demand points dataset
# This dataset will be used for demand side in optimization and distance calculation.
# --------------------------------------------------------------------------------

write.csv(final_data, "Demand_points.csv", row.names = FALSE)

# --------------------------------------------------------------------------------
# STEP 5: Aggregate demand by district to identify potential warehouse hub locations
# For each district, we calculate total demand and the average geographic center.
# --------------------------------------------------------------------------------

district_data <- final_data %>%
  group_by(district) %>%
  summarise(
    TotalDemand = sum(Demand, na.rm = TRUE),  # Sum all demand for the district
    Latitude = mean(latitude, na.rm = TRUE),  # Average lat-long as center of district
    Longitude = mean(longitude, na.rm = TRUE)
  )

# --------------------------------------------------------------------------------
# STEP 6: Select the top 20 districts by demand as possible warehouse candidates
# These are the highest-demand areas where warehouses are likely to be effective.
# --------------------------------------------------------------------------------

hub_candidates <- district_data %>%
  arrange(desc(TotalDemand)) %>%  # Sort districts by descending demand
  slice(1:20)  # Select the top 20 as warehouse hub candidates

# --------------------------------------------------------------------------------
# STEP 7: Save the selected warehouse candidate locations to CSV
# This file will be used in the optimization model to define candidate sites.
# --------------------------------------------------------------------------------

write.csv(hub_candidates, "Warehouse_points.csv", row.names = FALSE)


Warning message:
“One or more parsing issues, call `problems()` on your data frame for details,
e.g.:
  dat <- vroom(...)
  problems(dat)”
Warning message in scan(file = file, what = what, sep = sep, quote = quote, dec = dec, :
“EOF within quoted string”


In [3]:
# ------------------------------------------------------------------------
# Load necessary packages
# ------------------------------------------------------------------------
library(tidyverse)  # Includes dplyr, purrr, readr etc.
library(purrr)      # Specifically for pmap() to process rows with proper types

# ------------------------------------------------------------------------
# STEP 1: Read the processed demand and warehouse datasets
# ------------------------------------------------------------------------
demand_data <- read_csv("Demand_points.csv", show_col_types = FALSE)
warehouse_data <- read_csv("Warehouse_points.csv", show_col_types = FALSE)

# ------------------------------------------------------------------------
# STEP 2: Define the Haversine formula
# Calculates great-circle distance between two lat-long points on Earth
# ------------------------------------------------------------------------
haversine <- function(lat1, lon1, lat2, lon2) {
  R <- 6371  # Radius of Earth in kilometers
  dlat <- (lat2 - lat1) * pi / 180
  dlon <- (lon2 - lon1) * pi / 180
  lat1 <- lat1 * pi / 180
  lat2 <- lat2 * pi / 180

  a <- sin(dlat / 2)^2 + cos(lat1) * cos(lat2) * sin(dlon / 2)^2
  c <- 2 * atan2(sqrt(a), sqrt(1 - a))

  R * c  # Final output in kilometers
}

# ------------------------------------------------------------------------
# STEP 3: Calculate distances from each demand point to all warehouses
# We use pmap() to process each demand point row (lat1, lon1)
# For each, we apply mapply() to compute distances to all warehouse locations
# ------------------------------------------------------------------------

# This adds a new column: list of distances for each row in demand_data
dist_matrix_values <- demand_data %>%
  mutate(
    dist_vector = pmap(
      list(latitude, longitude),  # Input lat1, lon1 from demand point
      function(lat1, lon1) {
        mapply(function(lat2, lon2) {
          haversine(as.numeric(lat1), as.numeric(lon1),
                    as.numeric(lat2), as.numeric(lon2))
        }, warehouse_data$Latitude, warehouse_data$Longitude)
      }
    )
  )

# ------------------------------------------------------------------------
# STEP 4: Convert list-column of vectors into a proper distance matrix
# Each row corresponds to a demand point, each column is a warehouse
# ------------------------------------------------------------------------

# Combine the vectors into a full matrix (rows = demand points, cols = warehouses)
distance_matrix <- dist_matrix_values$dist_vector %>%
  do.call(rbind, .)

# Assign column names using warehouse district names
colnames(distance_matrix) <- warehouse_data$district

# ------------------------------------------------------------------------
# STEP 5: Merge the distance matrix with basic demand info (pincode, district)
# ------------------------------------------------------------------------

# Bind the distance matrix to the demand point identifiers
final_dist_matrix <- bind_cols(
  demand_data %>% select(pincode, district),
  as.data.frame(distance_matrix)
)

# ------------------------------------------------------------------------
# STEP 6: Save the final matrix to a CSV
# This file will be used in the Python optimization model
# ------------------------------------------------------------------------
write.csv(final_dist_matrix, "Distance_Matrix.csv", row.names = FALSE)


In [ ]:
distance_df <- read.csv("Distance_Matrix.csv")

# Exclude the first two columns ('pincode', 'district') to focus on distance values
max_distance <- max(distance_df[ , !(names(distance_df) %in% c("pincode", "district"))], na.rm = TRUE)

print(max_distance)


[1] 18391.7
